In [1]:
# Panggil fail solution.py yang kita buat tadi
from solution import STATES, get_group_states

# Tukar nombor dalam kurungan petak ini kepada nombor matrik sebenar ahli kumpulan anda!
nombor_matrik_kumpulan = [311279, 311294, 311397, 312114, 313429]

# Jalankan arahan untuk cari negeri
GROUP_STATES = get_group_states(nombor_matrik_kumpulan, STATES)

# Paparkan hasil negeri fokus
print("Negeri fokus kami ialah:", GROUP_STATES)

# Task 1
import json
from solution import validate_records, SURVEY_YEARS

# 1. Buka fail JSON dan ambil senarai di dalam kunci "records"
with open("data/raw/hh_income_state.json", "r") as f:
    data_inc = json.load(f)
    income_records = data_inc["records"] if isinstance(data_inc, dict) and "records" in data_inc else data_inc

with open("data/raw/hh_poverty_state.json", "r") as f:
    data_pov = json.load(f)
    poverty_records = data_pov["records"] if isinstance(data_pov, dict) and "records" in data_pov else data_pov

# 2. Tentukan kunci wajib
income_keys = ("state", "date", "income_median", "income_mean")
poverty_keys = ("state", "date", "poverty_absolute")

# 3. Jalankan fungsi semakan validate_records
validate_records(income_records, income_keys)
validate_records(poverty_records, poverty_keys)
print("Validation berjaya! Tiada ralat atau rekod bertindih dijumpai.")

# 4. Laporan Bilangan Rekod & Julat Tahun
years_income = sorted(list(set(int(r["date"][:4]) for r in income_records)))
print(f"\n--- Laporan Data Pendapatan ---")
print(f"• Jumlah rekod: {len(income_records)}")
print(f"• Tahun pertama: {years_income[0]} | Tahun terakhir: {years_income[-1]}")
print(f"• Bilangan tahun unik: {len(years_income)}")

# 5. Kira bilangan nilai None
none_counts_income = {key: sum(1 for r in income_records if r.get(key) is None) for key in income_keys}
print(f"• Bilangan nilai None dalam pendapatan: {none_counts_income}")

# 6. Tapis rekod untuk SURVEY_YEARS [2016, 2019, 2020, 2022, 2024]
filtered_income = [r for r in income_records if int(r["date"][:4]) in SURVEY_YEARS]

# 7. Semak kesemua 16 negeri wujud bagi setiap tahun tinjauan
for year in SURVEY_YEARS:
    states_in_year = set(r["state"] for r in filtered_income if int(r["date"][:4]) == year)
    if len(states_in_year) != 16:
        raise ValueError(f"Tahun {year} tidak mempunyai 16 negeri yang lengkap!")

print("Semakan lengkap: Kesemua 16 negeri wujud bagi setiap tahun dalam SURVEY_YEARS.")

Negeri fokus kami ialah: ['W.P. Putrajaya', 'W.P. Labuan', 'Pahang', 'Kelantan', 'Perak']
Validation berjaya! Tiada ralat atau rekod bertindih dijumpai.

--- Laporan Data Pendapatan ---
• Jumlah rekod: 319
• Tahun pertama: 1970 | Tahun terakhir: 2024
• Bilangan tahun unik: 22
• Bilangan nilai None dalam pendapatan: {'state': 0, 'date': 0, 'income_median': 11, 'income_mean': 0}
Semakan lengkap: Kesemua 16 negeri wujud bagi setiap tahun dalam SURVEY_YEARS.


In [2]:
# Task 2: Bina Matriks NumPy
import numpy as np
from solution import STATES, SURVEY_YEARS, POVERTY_YEARS, index_by_key, join_records, to_float

joined_recs, income_only, poverty_only = join_records(income_records, poverty_records)
print(f"Jumlah rekod berjaya dipadankan: {len(joined_recs)}")
print(f"Kunci hanya dalam pendapatan: {len(income_only)}")
print(f"Kunci hanya dalam kemiskinan: {len(poverty_only)}")

income_indexed = index_by_key(income_records)
poverty_indexed = index_by_key(poverty_records)

median_list = []
for state in STATES:
    row = []
    for year in SURVEY_YEARS:
        rec = income_indexed.get((state, year))
        val = rec["income_median"] if rec else None
        row.append(to_float(val))
    median_list.append(row)
MEDIAN = np.array(median_list)

mean_list = []
for state in STATES:
    row = []
    for year in SURVEY_YEARS:
        rec = income_indexed.get((state, year))
        val = rec["income_mean"] if rec else None
        row.append(to_float(val))
    mean_list.append(row)
MEAN = np.array(mean_list)

poverty_list = []
for state in STATES:
    row = []
    for year in POVERTY_YEARS:
        rec = poverty_indexed.get((state, year))
        val = rec.get("poverty_absolute") if rec else None
        row.append(to_float(val))
    poverty_list.append(row)
POVERTY = np.array(poverty_list)

print("\n--- Bentuk Matriks Task 2 ---")
print("MEDIAN shape:", MEDIAN.shape)
print("MEAN shape:", MEAN.shape)
print("POVERTY shape:", POVERTY.shape)

Jumlah rekod berjaya dipadankan: 307
Kunci hanya dalam pendapatan: 12
Kunci hanya dalam kemiskinan: 3

--- Bentuk Matriks Task 2 ---
MEDIAN shape: (16, 5)
MEAN shape: (16, 5)
POVERTY shape: (16, 4)
